In [ ]:
!pip install geopandas mapclassify

In [ ]:
# Cell 1 — install any missing libraries
!pip install plotly

In [ ]:
# Cell 2 — upload the CSV
from google.colab import files
uploaded = files.upload()

In [ ]:
"""
Food Desert Analysis Pipeline
USDA Food Access Research Atlas
HTML MAP
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────
# STEP 1: LOAD & CLEAN
# ─────────────────────────────────────────────────────────────
print("=" * 55)
print("STEP 1: Load & Clean")
print("=" * 55)

df = pd.read_csv('Food Access Research Atlas (1).csv')

features = [
    'PovertyRate',        # % of people below poverty line
    'MedianFamilyIncome', # median family income
    'Urban',              # urban (1) vs rural (0)
    'TractHUNV',          # households with no vehicle
    'TractSNAP',          # households on SNAP/food stamps
    'TractBlack',         # Black population count
    'TractHispanic',      # Hispanic population count
    'TractWhite',         # White population count
    'TractAsian',         # Asian population count
    'TractKids',          # children population
    'TractSeniors',       # seniors population
    'Pop2010',            # total population
    'LowIncomeTracts',    # binary: low income tract flag
    'GroupQuartersFlag',  # binary: group quarters (prisons, dorms)
]
target = 'LILATracts_1And10'  # USDA official food desert flag

df_clean = df[features + [target, 'CensusTract', 'State', 'County']].dropna()
print(f"Rows after cleaning: {len(df_clean):,} / {len(df):,}")
print(f"Food deserts: {df_clean[target].sum():,} ({df_clean[target].mean()*100:.1f}% of tracts)")


# ─────────────────────────────────────────────────────────────
# STEP 2: TRAIN / TEST SPLIT
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 2: Train/Test Split")
print("=" * 55)

X = df_clean[features]
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")


# ─────────────────────────────────────────────────────────────
# STEP 3: RANDOM FOREST — FEATURE SELECTION
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 3: Random Forest Feature Selection")
print("=" * 55)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf.fit(X_train, y_train)

importances = pd.Series(
    rf.feature_importances_, index=features
).sort_values(ascending=False)

print("\nFeature importances (ranked):")
for feat, val in importances.items():
    bar = '█' * int(val * 80)
    print(f"  {feat:<25} {val:.4f}  {bar}")

top_features = importances.head(10).index.tolist()
print(f"\nTop 10 features selected: {top_features}")


# ─────────────────────────────────────────────────────────────
# STEP 4: LOGISTIC REGRESSION ON TOP FEATURES
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 4: Logistic Regression")
print("=" * 55)

X_train_top = X_train[top_features]
X_test_top  = X_test[top_features]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_top)
X_test_scaled  = scaler.transform(X_test_top)

lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'   # important: handles imbalanced classes
)
lr.fit(X_train_scaled, y_train)

print("\nClassification Report:")
print(classification_report(y_test, lr.predict(X_test_scaled)))


# ─────────────────────────────────────────────────────────────
# STEP 5: EXTRACT GRADIENT SCORES PER TRACT
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 5: Extract Gradient Scores")
print("=" * 55)

X_full_scaled = scaler.transform(df_clean[top_features])
df_clean = df_clean.copy()
df_clean['food_desert_score'] = lr.predict_proba(X_full_scaled)[:, 1]

scores_df = df_clean[['CensusTract', 'State', 'County', 'food_desert_score', target]]
scores_df.to_csv('tract_scores.csv', index=False)
print(f"Saved tract_scores.csv — {len(scores_df):,} tracts")
print(f"Score range: {scores_df['food_desert_score'].min():.3f} → {scores_df['food_desert_score'].max():.3f}")
print(f"Mean score:  {scores_df['food_desert_score'].mean():.3f}")


# ─────────────────────────────────────────────────────────────
# STEP 6: VISUALIZATION — FEATURE PLOTS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 6: Feature Analysis Plots")
print("=" * 55)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0f1117')

# Feature importance
ax1 = axes[0]
ax1.set_facecolor('#1a1d2e')
colors_imp = ['#e05c5c' if i < 5 else '#5c8ee0' for i in range(len(importances))]
importances.plot(kind='barh', ax=ax1, color=colors_imp[::-1])
ax1.set_title('Random Forest Feature Importances', color='white', fontsize=13, pad=12)
ax1.set_xlabel('Importance', color='white')
ax1.tick_params(colors='white')
for spine in ax1.spines.values():
    spine.set_color('#333')
for label in ax1.get_yticklabels():
    label.set_color('white')

# LR coefficients
ax2 = axes[1]
ax2.set_facecolor('#1a1d2e')
coefs = pd.Series(lr.coef_[0], index=top_features).sort_values()
bar_colors = ['#e05c5c' if v > 0 else '#5ce07a' for v in coefs]
coefs.plot(kind='barh', ax=ax2, color=bar_colors)
ax2.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
ax2.set_title('Logistic Regression Coefficients\n(red = drives food desert risk)', color='white', fontsize=13, pad=12)
ax2.set_xlabel('Coefficient', color='white')
ax2.tick_params(colors='white')
for spine in ax2.spines.values():
    spine.set_color('#333')
for label in ax2.get_yticklabels():
    label.set_color('white')

plt.tight_layout(pad=2)
plt.savefig('feature_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
print("Saved feature_analysis.png")


# ─────────────────────────────────────────────────────────────
# STEP 7: GEOSPATIAL MAP
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 7: Geospatial Map")
print("=" * 55)

# Aggregate tract scores to county level using FIPS
scores_df = pd.read_csv('tract_scores.csv')
scores_df['county_fips'] = scores_df['CensusTract'].astype(str).str.zfill(11).str[:5]

county = scores_df.groupby('county_fips').agg(
    avg_food_desert_score=('food_desert_score', 'mean'),
    pct_food_desert_tracts=('LILATracts_1And10', 'mean'),
    tract_count=('food_desert_score', 'count'),
    State=('State', 'first'),
    County=('County', 'first')
).reset_index()

county['pct_food_desert_tracts'] = (county['pct_food_desert_tracts'] * 100).round(1)
county['avg_food_desert_score']  = county['avg_food_desert_score'].round(3)

fig_map = px.choropleth(
    county,
    geojson="https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json",
    locations='county_fips',
    color='avg_food_desert_score',
    color_continuous_scale=[
        [0.0, '#0d2137'],
        [0.2, '#1a4a6b'],
        [0.4, '#c47a00'],
        [0.6, '#e05500'],
        [0.8, '#c0392b'],
        [1.0, '#7b0000']
    ],
    scope='usa',
    labels={'avg_food_desert_score': 'Food Desert Score'},
    hover_data={
        'county_fips': False,
        'State': True,
        'County': True,
        'avg_food_desert_score': ':.3f',
        'pct_food_desert_tracts': True,
        'tract_count': True
    },
    title='U.S. Food Desert Risk Score by County'
)

fig_map.update_layout(
    title_font_size=20,
    title_x=0.5,
    geo=dict(showlakes=True, lakecolor='#0d2137', bgcolor='#0d2137', landcolor='#1a1d2e'),
    paper_bgcolor='#0d2137',
    font_color='white',
    coloraxis_colorbar=dict(
        title=dict(text='Food Desert<br>Risk Score', font=dict(color='white')),
        tickfont=dict(color='white'),
        len=0.6
    ),
    margin=dict(l=0, r=0, t=50, b=0),
    height=600
)

fig_map.write_html('food_desert_map.html')
print("Saved food_desert_map.html")
print("\n✓ Pipeline complete!")

In [ ]:
"""
Food Desert Analysis Pipeline — Interactive Folium Map
USDA Food Access Research Atlas
"""

# ─────────────────────────────────────────────────────────────
# INSTALL DEPENDENCIES (run this cell first in Colab)
# !pip install geopandas folium mapclassify -q
# ─────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import folium
import warnings
import zipfile
import urllib.request
import os
warnings.filterwarnings('ignore')


# ─────────────────────────────────────────────────────────────
# STEP 1: LOAD & CLEAN
# ─────────────────────────────────────────────────────────────
print("=" * 55)
print("STEP 1: Load & Clean")
print("=" * 55)

df = pd.read_csv('Food Access Research Atlas (1).csv')

features = [
    'PovertyRate',        # % of people below poverty line
    'MedianFamilyIncome', # median family income
    'Urban',              # urban (1) vs rural (0)
    'TractHUNV',          # households with no vehicle
    'TractSNAP',          # households on SNAP/food stamps
    'TractBlack',         # Black population count
    'TractHispanic',      # Hispanic population count
    'TractWhite',         # White population count
    'TractAsian',         # Asian population count
    'TractKids',          # children population
    'TractSeniors',       # seniors population
    'Pop2010',            # total population
    'LowIncomeTracts',    # binary: low income tract flag
    'GroupQuartersFlag',  # binary: group quarters (prisons, dorms)
]
target = 'LILATracts_1And10'

df_clean = df[features + [target, 'CensusTract', 'State', 'County']].dropna()
print(f"Rows after cleaning: {len(df_clean):,} / {len(df):,}")
print(f"Food deserts: {df_clean[target].sum():,} ({df_clean[target].mean()*100:.1f}% of tracts)")


# ─────────────────────────────────────────────────────────────
# STEP 2: TRAIN / TEST SPLIT
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 2: Train/Test Split")
print("=" * 55)

X = df_clean[features]
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")


# ─────────────────────────────────────────────────────────────
# STEP 3: RANDOM FOREST — FEATURE SELECTION
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 3: Random Forest Feature Selection")
print("=" * 55)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)
rf.fit(X_train, y_train)

importances = pd.Series(
    rf.feature_importances_, index=features
).sort_values(ascending=False)

print("\nFeature importances (ranked):")
for feat, val in importances.items():
    bar = '█' * int(val * 80)
    print(f"  {feat:<25} {val:.4f}  {bar}")

top_features = importances.head(10).index.tolist()
print(f"\nTop 10 features selected: {top_features}")


# ─────────────────────────────────────────────────────────────
# STEP 4: LOGISTIC REGRESSION ON TOP FEATURES
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 4: Logistic Regression")
print("=" * 55)

X_train_top = X_train[top_features]
X_test_top  = X_test[top_features]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_top)
X_test_scaled  = scaler.transform(X_test_top)

lr = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced'
)
lr.fit(X_train_scaled, y_train)

print("\nClassification Report:")
print(classification_report(y_test, lr.predict(X_test_scaled)))


# ─────────────────────────────────────────────────────────────
# STEP 4B: MODEL PERFORMANCE BREAKDOWN
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 4B: Model Performance Breakdown")
print("=" * 55)

y_pred = lr.predict(X_test_scaled)
y_prob = lr.predict_proba(X_test_scaled)[:, 1]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.patch.set_facecolor('#0f1117')

# ── Confusion Matrix ──────────────────────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#1a1d2e')
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Reds', ax=ax1,
    xticklabels=['Not Desert', 'Food Desert'],
    yticklabels=['Not Desert', 'Food Desert'],
    cbar=False, linewidths=0.5
)
ax1.set_title('Confusion Matrix', color='white', fontsize=13, pad=12)
ax1.set_xlabel('Predicted', color='white')
ax1.set_ylabel('Actual', color='white')
ax1.tick_params(colors='white')
for label in ax1.get_xticklabels() + ax1.get_yticklabels():
    label.set_color('white')

tn, fp, fn, tp = cm.ravel()
print(f"True Positives  (correctly caught food deserts): {tp:,}")
print(f"False Negatives (missed food deserts):           {fn:,}")
print(f"False Positives (wrongly flagged as desert):     {fp:,}")
print(f"True Negatives  (correctly cleared):             {tn:,}")

# ── ROC Curve ────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#1a1d2e')
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
ax2.plot(fpr, tpr, color='#e05c5c', linewidth=2, label=f'AUC = {auc:.3f}')
ax2.plot([0, 1], [0, 1], color='grey', linestyle='--', linewidth=1)
ax2.set_title('ROC Curve', color='white', fontsize=13, pad=12)
ax2.set_xlabel('False Positive Rate', color='white')
ax2.set_ylabel('True Positive Rate', color='white')
ax2.tick_params(colors='white')
ax2.legend(facecolor='#1a1d2e', labelcolor='white', fontsize=11)
for spine in ax2.spines.values():
    spine.set_color('#333')
print(f"\nROC-AUC Score: {auc:.3f}  (1.0 = perfect, 0.5 = random guessing)")

# ── Precision-Recall Curve ───────────────────────────────────
ax3 = axes[2]
ax3.set_facecolor('#1a1d2e')
precision, recall, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)
ax3.plot(recall, precision, color='#5c8ee0', linewidth=2, label=f'AP = {ap:.3f}')
ax3.axhline(y=y_test.mean(), color='grey', linestyle='--', linewidth=1,
            label=f'Baseline = {y_test.mean():.2f}')
ax3.set_title('Precision-Recall Curve', color='white', fontsize=13, pad=12)
ax3.set_xlabel('Recall', color='white')
ax3.set_ylabel('Precision', color='white')
ax3.tick_params(colors='white')
ax3.legend(facecolor='#1a1d2e', labelcolor='white', fontsize=11)
for spine in ax3.spines.values():
    spine.set_color('#333')
print(f"Average Precision Score: {ap:.3f}  (1.0 = perfect, {y_test.mean():.2f} = random)")

plt.suptitle('Model Performance — Food Desert Classification',
             color='white', fontsize=15, y=1.02)
plt.tight_layout(pad=2)
plt.savefig('model_performance.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
print("\nSaved model_performance.png")
plt.show()


# ─────────────────────────────────────────────────────────────
# STEP 5: EXTRACT GRADIENT SCORES PER TRACT
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 5: Extract Gradient Scores")
print("=" * 55)

X_full_scaled = scaler.transform(df_clean[top_features])
df_clean = df_clean.copy()
df_clean['food_desert_score'] = lr.predict_proba(X_full_scaled)[:, 1]
df_clean['GEOID'] = df_clean['CensusTract'].astype(str).str.zfill(11)

scores_df = df_clean[['GEOID', 'CensusTract', 'State', 'County', 'food_desert_score', target]]
scores_df.to_csv('tract_scores.csv', index=False)
print(f"Saved tract_scores.csv — {len(scores_df):,} tracts")
print(f"Score range: {scores_df['food_desert_score'].min():.3f} → {scores_df['food_desert_score'].max():.3f}")


# ─────────────────────────────────────────────────────────────
# STEP 6: FEATURE ANALYSIS PLOTS
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 6: Feature Analysis Plots")
print("=" * 55)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('#0f1117')

# Feature importance
ax1 = axes[0]
ax1.set_facecolor('#1a1d2e')
colors_imp = ['#e05c5c' if i < 5 else '#5c8ee0' for i in range(len(importances))]
importances.plot(kind='barh', ax=ax1, color=colors_imp[::-1])
ax1.set_title('Random Forest Feature Importances', color='white', fontsize=13, pad=12)
ax1.set_xlabel('Importance', color='white')
ax1.tick_params(colors='white')
for spine in ax1.spines.values():
    spine.set_color('#333')
for label in ax1.get_yticklabels():
    label.set_color('white')

# LR coefficients
ax2 = axes[1]
ax2.set_facecolor('#1a1d2e')
coefs = pd.Series(lr.coef_[0], index=top_features).sort_values()
bar_colors = ['#e05c5c' if v > 0 else '#5ce07a' for v in coefs]
coefs.plot(kind='barh', ax=ax2, color=bar_colors)
ax2.axvline(0, color='white', linewidth=0.8, linestyle='--', alpha=0.5)
ax2.set_title('Logistic Regression Coefficients\n(red = drives food desert risk)',
              color='white', fontsize=13, pad=12)
ax2.set_xlabel('Coefficient', color='white')
ax2.tick_params(colors='white')
for spine in ax2.spines.values():
    spine.set_color('#333')
for label in ax2.get_yticklabels():
    label.set_color('white')

plt.tight_layout(pad=2)
plt.savefig('feature_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0f1117')
print("Saved feature_analysis.png")
plt.show()


# ─────────────────────────────────────────────────────────────
# STEP 7: INTERACTIVE FOLIUM MAP (via GeoPandas)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 55)
print("STEP 7: Interactive Folium Map")
print("=" * 55)

# Download Census TIGER tract shapefile if not already present
shp_zip  = 'cb_2020_us_tract_500k.zip'
shp_dir  = 'tracts_shp'
shp_file = f'{shp_dir}/cb_2020_us_tract_500k.shp'

if not os.path.exists(shp_file):
    print("Downloading Census TIGER tract shapefile (~45MB)...")
    urllib.request.urlretrieve(
        "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_tract_500k.zip",
        shp_zip
    )
    print("Unzipping...")
    with zipfile.ZipFile(shp_zip, 'r') as z:
        z.extractall(shp_dir)
    print("Done.")
else:
    print("Shapefile already exists, skipping download.")

# Load shapefile and merge scores
print("Loading shapefile...")
gdf = gpd.read_file(shp_file)
gdf['GEOID'] = gdf['GEOID'].astype(str).str.zfill(11)

scores_df = pd.read_csv('tract_scores.csv')
scores_df['GEOID'] = scores_df['GEOID'].astype(str).str.zfill(11)

gdf = gdf.merge(
    scores_df[['GEOID', 'food_desert_score', 'LILATracts_1And10', 'State', 'County']],
    on='GEOID', how='left'
)
print(f"Matched {gdf['food_desert_score'].notna().sum():,} tracts with scores")

# Continental US only, reproject to lat/lon for Folium
gdf_cont = gdf[~gdf['STATEFP'].isin(['02', '15', '72'])].copy()
gdf_cont = gdf_cont.to_crs("EPSG:4326")

# ── Aggregate to county level ─────────────────────────────────
# 70k tract polygons crash the browser — county level (3,100) renders fast
print("Aggregating to county level...")
gdf_cont['county_fips'] = gdf_cont['GEOID'].str[:5]
gdf_cont = gdf_cont[gdf_cont['food_desert_score'].notna()].copy()

# Simplify geometry for faster rendering
gdf_cont['geometry'] = gdf_cont['geometry'].simplify(
    tolerance=0.01, preserve_topology=True
)

county_gdf = gdf_cont.dissolve(by='county_fips', aggfunc={
    'food_desert_score': 'mean',
    'LILATracts_1And10': 'mean',
    'State': 'first',
    'County': 'first'
}).reset_index()

county_gdf['food_desert_score'] = county_gdf['food_desert_score'].round(3)
county_gdf['pct_food_desert']   = (county_gdf['LILATracts_1And10'] * 100).round(1)
print(f"Counties ready: {len(county_gdf):,}")

# ── Build Folium map ──────────────────────────────────────────
print("Building interactive map...")
m = folium.Map(
    location=[39.5, -98.35],
    zoom_start=5,
    tiles='CartoDB dark_matter'
)

# Choropleth color layer
folium.Choropleth(
    geo_data=county_gdf.__geo_interface__,
    data=county_gdf,
    columns=['county_fips', 'food_desert_score'],
    key_on='feature.properties.county_fips',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.1,
    line_color='white',
    legend_name='Food Desert Risk Score (0 = low risk, 1 = high risk)',
    nan_fill_color='#1a1a1a',
).add_to(m)

# Invisible hover layer with tooltips
folium.GeoJson(
    county_gdf[[
        'county_fips', 'food_desert_score',
        'pct_food_desert', 'State', 'County', 'geometry'
    ]],
    style_function=lambda x: {'fillOpacity': 0, 'weight': 0},
    tooltip=folium.GeoJsonTooltip(
        fields=['State', 'County', 'food_desert_score', 'pct_food_desert'],
        aliases=[
            'State:',
            'County:',
            'Avg Risk Score:',
            '% Tracts Flagged as Food Desert:'
        ],
        localize=True,
        sticky=True,
        style=(
            "background-color: #1a1d2e; color: white; "
            "font-family: monospace; font-size: 13px; "
            "border: 1px solid #444; border-radius: 4px;"
        )
    )
).add_to(m)

m.save('food_desert_interactive.html')
print("Saved food_desert_interactive.html")

# ── Download all outputs ──────────────────────────────────────
from google.colab import files
files.download('food_desert_interactive.html')
files.download('model_performance.png')
files.download('feature_analysis.png')
files.download('tract_scores.csv')

print("\n✓ Pipeline complete! 4 files downloading...")
print("  → Open food_desert_interactive.html in Chrome or Firefox")

STEP 1: Load & Clean


FileNotFoundError: [Errno 2] No such file or directory: 'Food Access Research Atlas (1).csv'